In [1]:
import pandas as pd
import numpy as np 

In [ ]:
df = pd.read_csv("Games.csv", low_memory=False) # here, low memory = false inspects the dataset more carefully before assigning column data types.
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 73252 entries, 0 to 73251
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gameId            73252 non-null  int64  
 1   gameDateTimeEst   73252 non-null  str    
 2   hometeamCity      73245 non-null  str    
 3   hometeamName      73252 non-null  str    
 4   hometeamId        73252 non-null  int64  
 5   awayteamCity      73245 non-null  str    
 6   awayteamName      73252 non-null  str    
 7   awayteamId        73252 non-null  int64  
 8   homeScore         73252 non-null  int64  
 9   awayScore         73252 non-null  int64  
 10  winner            73252 non-null  int64  
 11  gameType          73252 non-null  str    
 12  gameSubtype       74 non-null     str    
 13  gameLabel         4016 non-null   str    
 14  gameSubLabel      301 non-null    str    
 15  seriesGameNumber  5796 non-null   str    
 16  attendance        1365 non-null   float64
 17  aren

In [ ]:
df["gameDateTimeEst"] = pd.to_datetime(
    df["gameDateTimeEst"],
    errors="coerce" # invalid dates become nan
)

df["gameDate"] = pd.to_datetime(
    df["gameDate"],
    errors="coerce"
) 


We converted the data into date columns because it allows us to easily extract features such as the day of the week, month, and year, which can be useful for our model. Additionally, having the date in a proper format allows us to perform time-based analysis and comparisons between different games and seasons.

In [ ]:
df["season_year"] = df["gameDateTimeEst"].dt.year
# dt here extracts only the date from the datetime.
df = df[
    (df["season_year"] >= 2021) &   # keep seasons 2021 and later
    (df["season_year"] <= 2026)     # keep seasons up to 2026
].copy() # now all the previous seasons that we wont be using are removed. independetly stored as the data in the new dataframe.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8019 entries, 0 to 8018
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   gameId            8019 non-null   int64         
 1   gameDateTimeEst   8019 non-null   datetime64[us]
 2   hometeamCity      8012 non-null   str           
 3   hometeamName      8019 non-null   str           
 4   hometeamId        8019 non-null   int64         
 5   awayteamCity      8012 non-null   str           
 6   awayteamName      8019 non-null   str           
 7   awayteamId        8019 non-null   int64         
 8   homeScore         8019 non-null   int64         
 9   awayScore         8019 non-null   int64         
 10  winner            8019 non-null   int64         
 11  gameType          8019 non-null   str           
 12  gameSubtype       74 non-null     str           
 13  gameLabel         948 non-null    str           
 14  gameSubLabel      301 non-null    s

In [12]:
df = df.drop_duplicates()

df = df.drop_duplicates(
    subset=["gameId"]   # removes duplicate game IDs specifically
)
cleaned_data = df

with this our duplicated data dropped and the next step is to sort it based on oldes to newest date and then we will be ready to train our model.


In [ ]:
cleaned_data= cleaned_data.sort_values(
    "gameDateTimeEst"   
)



now wewill be resetting the index.

In [ ]:
cleaned_data = cleaned_data.reset_index(
    drop=True)
cleaned_data.info()


<class 'pandas.DataFrame'>
RangeIndex: 8019 entries, 0 to 8018
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   gameId            8019 non-null   int64         
 1   gameDateTimeEst   8019 non-null   datetime64[us]
 2   hometeamCity      8012 non-null   str           
 3   hometeamName      8019 non-null   str           
 4   hometeamId        8019 non-null   int64         
 5   awayteamCity      8012 non-null   str           
 6   awayteamName      8019 non-null   str           
 7   awayteamId        8019 non-null   int64         
 8   homeScore         8019 non-null   int64         
 9   awayScore         8019 non-null   int64         
 10  winner            8019 non-null   int64         
 11  gameType          8019 non-null   str           
 12  gameSubtype       74 non-null     str           
 13  gameLabel         948 non-null    str           
 14  gameSubLabel      301 non-null    s

NameError: name 'gameId' is not defined

we will be dropping incomplete games.

In [ ]:
cleaned_data = cleaned_data.dropna(
    subset=[
        "homeScore",
        "awayScore"
    ]
)

# there were some entries with negative scores in the data we are removing them
cleaned_data = cleaned_data[
    (cleaned_data["homeScore"] > 0) &
    (cleaned_data["awayScore"] > 0)
]



We will be doing small parts of feature engineering here. we will be making some names fro the variables.

In [22]:
cleaned_data["home_win"] = (
    cleaned_data["homeScore"] >
    cleaned_data["awayScore"]
).astype(int) # int convers true to 1 and false to 0. this is our target variable. 1 means home team wins and 0 means away team wins.
cleaned_data.info()

<class 'pandas.DataFrame'>
Index: 8017 entries, 0 to 8018
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   gameId            8017 non-null   int64         
 1   gameDateTimeEst   8017 non-null   datetime64[us]
 2   hometeamCity      8010 non-null   str           
 3   hometeamName      8017 non-null   str           
 4   hometeamId        8017 non-null   int64         
 5   awayteamCity      8010 non-null   str           
 6   awayteamName      8017 non-null   str           
 7   awayteamId        8017 non-null   int64         
 8   homeScore         8017 non-null   int64         
 9   awayScore         8017 non-null   int64         
 10  winner            8017 non-null   int64         
 11  gameType          8017 non-null   str           
 12  gameSubtype       74 non-null     str           
 13  gameLabel         948 non-null    str           
 14  gameSubLabel      301 non-null    str   

In [26]:
cleaned_data["home_team"] = (
    cleaned_data["hometeamCity"].astype(str)
    + " " +
    cleaned_data["hometeamName"].astype(str)
)
 # here we are creating a new column called home team by combining the city and the name of the team. this will be useful for our model to learn the team names and their performance.
cleaned_data["away_team"] = (
    cleaned_data["awayteamCity"].astype(str)
    + " " +
    cleaned_data["awayteamName"].astype(str)
) # here we are creating a new column called away team by combining the city and the name of the team. this will be useful for our model to learn the team names and their performance.
cleaned_data.info()
cleaned_data.head()
cleaned_data["home_team"].head()

<class 'pandas.DataFrame'>
Index: 8017 entries, 0 to 8018
Data columns (total 27 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   gameId            8017 non-null   int64         
 1   gameDateTimeEst   8017 non-null   datetime64[us]
 2   hometeamCity      8010 non-null   str           
 3   hometeamName      8017 non-null   str           
 4   hometeamId        8017 non-null   int64         
 5   awayteamCity      8010 non-null   str           
 6   awayteamName      8017 non-null   str           
 7   awayteamId        8017 non-null   int64         
 8   homeScore         8017 non-null   int64         
 9   awayScore         8017 non-null   int64         
 10  winner            8017 non-null   int64         
 11  gameType          8017 non-null   str           
 12  gameSubtype       74 non-null     str           
 13  gameLabel         948 non-null    str           
 14  gameSubLabel      301 non-null    str   

0          Dallas Mavericks
1           Detroit Pistons
2         Charlotte Hornets
3             Brooklyn Nets
4    Minnesota Timberwolves
Name: home_team, dtype: str

alright with this we have created the team name variables. properly. this was based on jack and my analysis based on the code. 

In [27]:
# this is what chat suggested me to do . clean the data by removing spaces.
cleaned_data["home_team"] = (
    cleaned_data["hometeamCity"].astype(str)
    + " " +
    cleaned_data["hometeamName"].astype(str)
)

cleaned_data["away_team"] = (
    cleaned_data["awayteamCity"].astype(str)
    + " " +
    cleaned_data["awayteamName"].astype(str)
)


now, we will be creating few features. We will be isolating normal games from playoffs due to their structure. we will also be removing covid era.

In [28]:
cleaned_data["is_playoff"] = (
    cleaned_data["gameType"] != "Regular Season"
).astype(int)
cleaned_data["covid_era"] = (
    cleaned_data["season_year"] <= 2021
).astype(int)
cleaned_data.head()


,gameId,gameDateTimeEst,hometeamCity,hometeamName,hometeamId,awayteamCity,awayteamName,awayteamId,homeScore,awayScore,...,arenaCity,arenaState,officials,gameDate,season_year,home_win,home_team,away_team,is_playoff,covid_era
0,22000071,2021-01-01 19:00:00,Dallas,Mavericks,1610612742,Miami,Heat,1610612748,93,83,...,NaN,NaN,NaN,2021-01-01 19:00:00,2021,1,Dallas Mavericks,Miami Heat,0,1
1,22000070,2021-01-01 19:00:00,Detroit,Pistons,1610612765,Boston,Celtics,1610612738,96,93,...,NaN,NaN,NaN,2021-01-01 19:00:00,2021,1,Detroit Pistons,Boston Celtics,0,1
2,22000069,2021-01-01 19:00:00,Charlotte,Hornets,1610612766,Memphis,Grizzlies,1610612763,93,108,...,NaN,NaN,NaN,2021-01-01 19:00:00,2021,0,Charlotte Hornets,Memphis Grizzlies,0,1
3,22000072,2021-01-01 19:30:00,Brooklyn,Nets,1610612751,Atlanta,Hawks,1610612737,96,114,...,NaN,NaN,NaN,2021-01-01 19:30:00,2021,0,Brooklyn Nets,Atlanta Hawks,0,1
4,22000074,2021-01-01 20:00:00,Minnesota,Timberwolves,1610612750,Washington,Wizards,1610612764,109,130,...,NaN,NaN,NaN,2021-01-01 20:00:00,2021,0,Minnesota Timberwolves,Washington Wizards,0,1


i have confirmed the code until now is working, i checked the dallas vs heat game online in the same date, so we should be good for now.